# RAG Base Conversational Agent

# 1. Loading Necessary Packages and Class



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -r /content/drive/MyDrive/rag_nov/clib/requirement.txt

In [ ]:
# Access HuggingFace Models
import os
huggingface_key = "your_hugging_face_key_here"
from huggingface_hub import login
login(huggingface_key)


In [4]:
# Config path
import sys

local = True

if local:
  path_for_file = "data/en/About_NRB.pdf"
  path_for_files = "data/en/"
  path_for_qa = "data/en/qa_en.xlsx"
  output_path = "output/"
  sys.path.insert(0, "clib")

else:
  path_for_file = "/content/drive/MyDrive/rag_nov/data/en/About_NRB.pdf"
  path_for_files = "/content/drive/MyDrive/rag_nov/data/en/"
  path_for_qa = "/content/drive/MyDrive/rag_nov/data/en/qa_en.xlsx"
  output_path = "/content/drive/MyDrive/rag_nov/output/"
  sys.path.insert(0, '/content/drive/MyDrive/rag_nov/clib')


In [5]:
# reloading
import importlib
import data_cleanner
import  data_loader
import  chunking
import retriever
import augmentation
import generator
import evaluate
import user_access
import user_access_exl
for module in [data_cleanner ,  data_loader ,  chunking , retriever , augmentation , generator , evaluate , user_access, user_access_exl]:
    importlib.reload(module)

In [6]:
from data_cleanner import DataCleanner
from data_loader import DataLoader
from chunking import Chunking
from retriever import Retriever
from augmentation import Augmentation
from generator import Generator
from evaluate import InformationRetrievalEvaluator, GeneratorEvaluator, Evaluation, GtmInformationRetrievalEvaluator
from user_access import UserAccess
from user_access_exl import UserAccessExl

# 2. Loading Previous Chunks,  Retriving and Evaluating

## 2.1 BAAI/bge-large-en-v1.5
---

In [5]:
useracess_exl = UserAccessExl(output_path, "BAAI/bge-large-en-v1.5")
response = useracess_exl.read_qa_from_excel(path_for_qa)

525 525 525


In [10]:
from tabulate import tabulate

def show_evaluation(df, chunk_type = 'overlapping', k = 5):
  k += 1
  result = []
  for i in range(1, k):
    result.append(GtmInformationRetrievalEvaluator(k=3).eval_retrieval(df, chunk_type, i))

  print(tabulate(result, headers="keys", tablefmt="grid"))



### 2.1.1 Overlapped Chunks

#### I. Sparse

In [7]:
df_overlapped_sparse = useracess_exl.overlapped_chunking_sparse_embed()
df_overlapped_sparse.to_excel(output_path + 'bge_large_overlapped_chunking_sparse_embed.xlsx')


c:\Users\G00248\OneDrive\11_23\rag_nov\clib\retriever.py:23: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  return self.bm25.get_relevant_documents(query)


Success


In [8]:
print("Result: Overlapped Chunking & Sparse Embedding")
show_evaluation(df_overlapped_sparse, chunk_type = 'overlapping')

Result: Overlapped Chunking & Sparse Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.607619 | 0.539048 | 0.561587 |
+-----+-------------+----------+----------+
|   2 |    0.388571 | 0.668095 | 0.481079 |
+-----+-------------+----------+----------+
|   3 |    0.283175 | 0.721175 | 0.39829  |
+-----+-------------+----------+----------+
|   4 |    0.223333 | 0.757841 | 0.338555 |
+-----+-------------+----------+----------+
|   5 |    0.183619 | 0.775683 | 0.291722 |
+-----+-------------+----------+----------+


#### II. Dense

In [9]:
df_overlapped_dense = useracess_exl.overlapped_chunking_dense_embed()
df_overlapped_dense.to_excel(output_path + 'bge_large_overlapped_chunking_dense_embed.xlsx')


Success


In [10]:
print("Result: Overlapped Chunking & Dense Embedding")
show_evaluation(df_overlapped_dense, chunk_type = 'overlapping')

Result: Overlapped Chunking & Dense Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.660952 | 0.581968 | 0.607619 |
+-----+-------------+----------+----------+
|   2 |    0.434286 | 0.736794 | 0.53385  |
+-----+-------------+----------+----------+
|   3 |    0.325079 | 0.821937 | 0.455814 |
+-----+-------------+----------+----------+
|   4 |    0.255714 | 0.861619 | 0.386707 |
+-----+-------------+----------+----------+
|   5 |    0.211048 | 0.883429 | 0.334457 |
+-----+-------------+----------+----------+


#### III. Hybrid

In [11]:
df_overlapped_hyrid = useracess_exl.overlapped_chunking_hyrid_embed()
df_overlapped_hyrid.to_excel(output_path + 'bge_large_overlapped_chunking_hybrid_embed.xlsx')

Success


In [12]:
print("Result: Overlapped Chunking & Hybrind Embedding")
show_evaluation(df_overlapped_dense, chunk_type = 'overlapping', k = 10)

Result: Overlapped Chunking & Hybrind Embedding


+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.660952 | 0.581968 | 0.607619 |
+-----+-------------+----------+----------+
|   2 |    0.434286 | 0.736794 | 0.53385  |
+-----+-------------+----------+----------+
|   3 |    0.325079 | 0.821937 | 0.455814 |
+-----+-------------+----------+----------+
|   4 |    0.255714 | 0.861619 | 0.386707 |
+-----+-------------+----------+----------+
|   5 |    0.211048 | 0.883429 | 0.334457 |
+-----+-------------+----------+----------+
|   6 |    0.175873 | 0.883429 | 0.288474 |
+-----+-------------+----------+----------+
|   7 |    0.150748 | 0.883429 | 0.253645 |
+-----+-------------+----------+----------+
|   8 |    0.131905 | 0.883429 | 0.226342 |
+-----+-------------+----------+----------+
|   9 |    0.117249 | 0.883429 | 0.204358 |
+-----+-------------+----------+----------+
|  10 |    0.105524 | 0.883429 | 0.186275 |
+-----+-------------+----------+

### 2.1.2 Recursive Chunking

#### I. Sparse



In [13]:
df_recursive_sparse = useracess_exl.recursive_chunking_sparse_embed()
df_recursive_sparse.to_excel(output_path + 'bge_large_recursive_chunking_sparse_embed.xlsx')


Success


In [14]:
print("Result: Recursive Chunking & Sparse Embedding")
show_evaluation(df_recursive_sparse, chunk_type = 'recursive', k = 5)

Result: Recursive Chunking & Sparse Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.641905 | 0.633333 | 0.63619  |
+-----+-------------+----------+----------+
|   2 |    0.381905 | 0.749524 | 0.504444 |
+-----+-------------+----------+----------+
|   3 |    0.274286 | 0.803492 | 0.407492 |
+-----+-------------+----------+----------+
|   4 |    0.212381 | 0.829206 | 0.337052 |
+-----+-------------+----------+----------+
|   5 |    0.174857 | 0.851746 | 0.289206 |
+-----+-------------+----------+----------+


#### II. Dense

In [15]:
df_recursive_dense = useracess_exl.recursive_chunking_dense_embed()
df_recursive_dense.to_excel(output_path + 'bge_large_recursive_chunking_dense_embed.xlsx')

Success


In [16]:
print("Result: Recursive Chunking & Dense Embedding")
show_evaluation(df_recursive_dense, chunk_type = 'recursive', k = 5)

Result: Recursive Chunking & Dense Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.685714 | 0.673333 | 0.67746  |
+-----+-------------+----------+----------+
|   2 |    0.424762 | 0.831429 | 0.560317 |
+-----+-------------+----------+----------+
|   3 |    0.299683 | 0.877143 | 0.445143 |
+-----+-------------+----------+----------+
|   4 |    0.234762 | 0.913968 | 0.372227 |
+-----+-------------+----------+----------+
|   5 |    0.191619 | 0.930794 | 0.316689 |
+-----+-------------+----------+----------+


#### III. Hybrid

In [17]:
df_recursive_hyrid = useracess_exl.recursive_chunking_hyrid_embed()
df_recursive_hyrid.to_excel(output_path + 'bge_large_recursive_chunking_hyrid_embed.xlsx')


Success


In [18]:
print("Result: Recursive Chunking & Hybrid Embedding")
show_evaluation(df_recursive_hyrid, chunk_type = 'recursive', k = 10)

Result: Recursive Chunking & Hybrid Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.691429 | 0.679048 | 0.683175 |
+-----+-------------+----------+----------+
|   2 |    0.386667 | 0.759048 | 0.510794 |
+-----+-------------+----------+----------+
|   3 |    0.263492 | 0.775238 | 0.39219  |
+-----+-------------+----------+----------+
|   4 |    0.199048 | 0.780952 | 0.316444 |
+-----+-------------+----------+----------+
|   5 |    0.16     | 0.784762 | 0.265215 |
+-----+-------------+----------+----------+
|   6 |    0.154921 | 0.912381 | 0.264354 |
+-----+-------------+----------+----------+
|   7 |    0.13551  | 0.930159 | 0.236095 |
+-----+-------------+----------+----------+
|   8 |    0.120476 | 0.941587 | 0.213172 |
+-----+-------------+----------+----------+
|   9 |    0.10709  | 0.941587 | 0.191937 |
+-----+-------------+----------+----------+
|  10 |    0.096381 | 0.941587

### 2.1.3 Semantic Chunking

#### I. Sparse

In [19]:
df_semantic_sparse = useracess_exl.semantic_chunking_sparse_embed()
df_semantic_sparse.to_excel(output_path + 'bge_large_semantic_chunking_sparse_embed.xlsx')

Success


In [20]:
print("Result: Semantic Chunking & Sparse Embedding")
show_evaluation(df_semantic_sparse, chunk_type = 'semantic', k = 5)

Result: Semantic Chunking & Sparse Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.619048 | 0.604444 | 0.609206 |
+-----+-------------+----------+----------+
|   2 |    0.378095 | 0.736508 | 0.497397 |
+-----+-------------+----------+----------+
|   3 |    0.272381 | 0.790794 | 0.403175 |
+-----+-------------+----------+----------+
|   4 |    0.214286 | 0.830794 | 0.339247 |
+-----+-------------+----------+----------+
|   5 |    0.174857 | 0.844762 | 0.288503 |
+-----+-------------+----------+----------+


### II. Dense

In [21]:
df_semantic_dense = useracess_exl.semantic_chunking_dense_embed()
df_semantic_dense.to_excel(output_path + 'bge_large_semantic_chunking_dense_embed.xlsx')

Success


In [22]:
print("Result: Semantic Chunking & Dense Embedding")
show_evaluation(df_semantic_dense, chunk_type = 'semantic', k = 5)

Result: Semantic Chunking & Dense Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.653333 | 0.634921 | 0.640952 |
+-----+-------------+----------+----------+
|   2 |    0.40381  | 0.778413 | 0.528508 |
+-----+-------------+----------+----------+
|   3 |    0.293968 | 0.846032 | 0.433651 |
+-----+-------------+----------+----------+
|   4 |    0.229524 | 0.880317 | 0.362104 |
+-----+-------------+----------+----------+
|   5 |    0.187429 | 0.897143 | 0.308458 |
+-----+-------------+----------+----------+


#### III. Hybrid

In [23]:
df_semantic_hybrid = useracess_exl.semantic_chunking_hyrid_embed()
df_semantic_hybrid.to_excel(output_path + 'bge_large_semantic_chunking_hybrid_embed.xlsx')

Success


In [24]:
print("Result: Semantic Chunking & Dense Embedding")
show_evaluation(df_semantic_hybrid, chunk_type = 'semantic', k = 10)

Result: Semantic Chunking & Dense Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |   0.645714  | 0.629206 | 0.634603 |
+-----+-------------+----------+----------+
|   2 |   0.364762  | 0.706032 | 0.478349 |
+-----+-------------+----------+----------+
|   3 |   0.247619  | 0.71746  | 0.366222 |
+-----+-------------+----------+----------+
|   4 |   0.18619   | 0.719365 | 0.294422 |
+-----+-------------+----------+----------+
|   5 |   0.151619  | 0.729841 | 0.249932 |
+-----+-------------+----------+----------+
|   6 |   0.150794  | 0.871429 | 0.256032 |
+-----+-------------+----------+----------+
|   7 |   0.132789  | 0.895238 | 0.230402 |
+-----+-------------+----------+----------+
|   8 |   0.11881   | 0.915238 | 0.209589 |
+-----+-------------+----------+----------+
|   9 |   0.106455  | 0.922857 | 0.190286 |
+-----+-------------+----------+----------+
|  10 |   0.0958095 | 0.922857 |

## 2.2 intfloat/e5-large-v2

---



In [25]:
useracess_exl = UserAccessExl(output_path, "intfloat/e5-large-v2")
response = useracess_exl.read_qa_from_excel(path_for_qa)

525 525 525


### 2.2.1 Overlapped Chunks

#### I. Sparse

In [26]:
df_overlapped_sparse = useracess_exl.overlapped_chunking_sparse_embed()
df_overlapped_sparse.to_excel(output_path + 'e5_large_overlapped_chunking_sparse_embed.xlsx')


Success


In [27]:
print("Result: Overlapped Chunking & Sparse Embedding")
show_evaluation(df_overlapped_sparse, chunk_type = 'overlapping')

Result: Overlapped Chunking & Sparse Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.607619 | 0.539048 | 0.561587 |
+-----+-------------+----------+----------+
|   2 |    0.388571 | 0.668095 | 0.481079 |
+-----+-------------+----------+----------+
|   3 |    0.283175 | 0.721175 | 0.39829  |
+-----+-------------+----------+----------+
|   4 |    0.223333 | 0.757841 | 0.338555 |
+-----+-------------+----------+----------+
|   5 |    0.183619 | 0.775683 | 0.291722 |
+-----+-------------+----------+----------+


#### II. Dense

In [28]:
df_overlapped_dense = useracess_exl.overlapped_chunking_dense_embed()
df_overlapped_dense.to_excel(output_path + 'e5_large_overlapped_chunking_dense_embed.xlsx')


Success


In [29]:
print("Result: Overlapped Chunking & Dense Embedding")
show_evaluation(df_overlapped_dense, chunk_type = 'overlapping')

Result: Overlapped Chunking & Dense Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.670476 | 0.592603 | 0.617905 |
+-----+-------------+----------+----------+
|   2 |    0.442857 | 0.755048 | 0.54566  |
+-----+-------------+----------+----------+
|   3 |    0.326984 | 0.826381 | 0.458109 |
+-----+-------------+----------+----------+
|   4 |    0.256667 | 0.860095 | 0.387194 |
+-----+-------------+----------+----------+
|   5 |    0.210667 | 0.882952 | 0.333882 |
+-----+-------------+----------+----------+


#### III. Hybrid **bold text**

In [30]:
df_overlapped_hyrid = useracess_exl.overlapped_chunking_hyrid_embed()
df_overlapped_hyrid.to_excel(output_path + 'e5_large_overlapped_chunking_hybrid_embed.xlsx')

Success


In [31]:
print("Result: Overlapped Chunking & Hybrind Embedding")
show_evaluation(df_overlapped_dense, chunk_type = 'overlapping', k = 10)

Result: Overlapped Chunking & Hybrind Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.670476 | 0.592603 | 0.617905 |
+-----+-------------+----------+----------+
|   2 |    0.442857 | 0.755048 | 0.54566  |
+-----+-------------+----------+----------+
|   3 |    0.326984 | 0.826381 | 0.458109 |
+-----+-------------+----------+----------+
|   4 |    0.256667 | 0.860095 | 0.387194 |
+-----+-------------+----------+----------+
|   5 |    0.210667 | 0.882952 | 0.333882 |
+-----+-------------+----------+----------+
|   6 |    0.175556 | 0.882952 | 0.28797  |
+-----+-------------+----------+----------+
|   7 |    0.150476 | 0.882952 | 0.253198 |
+-----+-------------+----------+----------+
|   8 |    0.131667 | 0.882952 | 0.22594  |
+-----+-------------+----------+----------+
|   9 |    0.117037 | 0.882952 | 0.203994 |
+-----+-------------+----------+----------+
|  10 |    0.105333 | 0.8829

### 2.2.2 Recursive Chunking


#### I. Sparse



In [32]:
df_recursive_sparse = useracess_exl.recursive_chunking_sparse_embed()
df_recursive_sparse.to_excel(output_path + 'e5_large_recursive_chunking_sparse_embed.xlsx')


Success


In [33]:
print("Result: Recursive Chunking & Sparse Embedding")
show_evaluation(df_recursive_sparse, chunk_type = 'recursive', k = 5)

Result: Recursive Chunking & Sparse Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.641905 | 0.633333 | 0.63619  |
+-----+-------------+----------+----------+
|   2 |    0.381905 | 0.749524 | 0.504444 |
+-----+-------------+----------+----------+
|   3 |    0.274286 | 0.803492 | 0.407492 |
+-----+-------------+----------+----------+
|   4 |    0.212381 | 0.829206 | 0.337052 |
+-----+-------------+----------+----------+
|   5 |    0.174857 | 0.851746 | 0.289206 |
+-----+-------------+----------+----------+


#### II. Dense

In [ ]:
df_recursive_dense = useracess_exl.recursive_chunking_dense_embed()
df_recursive_dense.to_excel(output_path + 'e5_large_recursive_chunking_dense_embed.xlsx')

Success


In [35]:
print("Result: Recursive Chunking & Dense Embedding")
show_evaluation(df_recursive_dense, chunk_type = 'recursive', k = 5)

Result: Recursive Chunking & Dense Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.708571 | 0.695238 | 0.699683 |
+-----+-------------+----------+----------+
|   2 |    0.438095 | 0.853968 | 0.576635 |
+-----+-------------+----------+----------+
|   3 |    0.311746 | 0.910794 | 0.462603 |
+-----+-------------+----------+----------+
|   4 |    0.242381 | 0.944127 | 0.384327 |
+-----+-------------+----------+----------+
|   5 |    0.199238 | 0.968889 | 0.329388 |
+-----+-------------+----------+----------+


#### III. Hybrid

In [36]:
df_recursive_hyrid = useracess_exl.recursive_chunking_hyrid_embed()
df_recursive_hyrid.to_excel(output_path + 'e5_large_recursive_chunking_hyrid_embed.xlsx')


Success


In [37]:
print("Result: Recursive Chunking & Hybrid Embedding")
show_evaluation(df_recursive_hyrid, chunk_type = 'recursive', k = 10)

Result: Recursive Chunking & Hybrid Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |   0.714286  | 0.700952 | 0.705397 |
+-----+-------------+----------+----------+
|   2 |   0.382857  | 0.750476 | 0.505397 |
+-----+-------------+----------+----------+
|   3 |   0.259683  | 0.761905 | 0.386095 |
+-----+-------------+----------+----------+
|   4 |   0.194762  | 0.761905 | 0.309333 |
+-----+-------------+----------+----------+
|   5 |   0.156952  | 0.766349 | 0.259796 |
+-----+-------------+----------+----------+
|   6 |   0.154286  | 0.905397 | 0.263008 |
+-----+-------------+----------+----------+
|   7 |   0.137143  | 0.936825 | 0.238688 |
+-----+-------------+----------+----------+
|   8 |   0.122143  | 0.951746 | 0.215973 |
+-----+-------------+----------+----------+
|   9 |   0.108571  | 0.951746 | 0.19447  |
+-----+-------------+----------+----------+
|  10 |   0.0977143 | 0.951746

### 2.2.3 Semantic Chunking

#### I. Sparse

In [38]:
df_semantic_sparse = useracess_exl.semantic_chunking_sparse_embed()
df_semantic_sparse.to_excel(output_path + 'e5_large_semantic_chunking_sparse_embed.xlsx')

Success


In [39]:
print("Result: Semantic Chunking & Sparse Embedding")
show_evaluation(df_semantic_sparse, chunk_type = 'semantic', k = 5)

Result: Semantic Chunking & Sparse Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.619048 | 0.604444 | 0.609206 |
+-----+-------------+----------+----------+
|   2 |    0.378095 | 0.736508 | 0.497397 |
+-----+-------------+----------+----------+
|   3 |    0.272381 | 0.790794 | 0.403175 |
+-----+-------------+----------+----------+
|   4 |    0.214286 | 0.830794 | 0.339247 |
+-----+-------------+----------+----------+
|   5 |    0.174857 | 0.844762 | 0.288503 |
+-----+-------------+----------+----------+


### II. Dense

In [40]:
df_semantic_dense = useracess_exl.semantic_chunking_dense_embed()
df_semantic_dense.to_excel(output_path + 'e5_large_semantic_chunking_dense_embed.xlsx')

Success


In [41]:
print("Result: Semantic Chunking & Dense Embedding")
show_evaluation(df_semantic_dense, chunk_type = 'semantic', k = 5)

Result: Semantic Chunking & Dense Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.660952 | 0.641587 | 0.647937 |
+-----+-------------+----------+----------+
|   2 |    0.405714 | 0.784127 | 0.531683 |
+-----+-------------+----------+----------+
|   3 |    0.292063 | 0.841905 | 0.431048 |
+-----+-------------+----------+----------+
|   4 |    0.230952 | 0.886667 | 0.364426 |
+-----+-------------+----------+----------+
|   5 |    0.189714 | 0.910476 | 0.312449 |
+-----+-------------+----------+----------+


#### III. Hybrid

In [42]:
df_semantic_hybrid = useracess_exl.semantic_chunking_hyrid_embed()
df_semantic_hybrid.to_excel(output_path + 'e5_large_semantic_chunking_hybrid_embed.xlsx')

Success


In [43]:
print("Result: Semantic Chunking & Dense Embedding")
show_evaluation(df_semantic_hybrid, chunk_type = 'semantic', k = 10)

Result: Semantic Chunking & Dense Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |   0.666667  | 0.648254 | 0.654286 |
+-----+-------------+----------+----------+
|   2 |   0.358095  | 0.694603 | 0.470095 |
+-----+-------------+----------+----------+
|   3 |   0.24254   | 0.704762 | 0.359048 |
+-----+-------------+----------+----------+
|   4 |   0.182857  | 0.708571 | 0.289379 |
+-----+-------------+----------+----------+
|   5 |   0.146667  | 0.710476 | 0.242154 |
+-----+-------------+----------+----------+
|   6 |   0.147302  | 0.855238 | 0.250385 |
+-----+-------------+----------+----------+
|   7 |   0.133333  | 0.901905 | 0.231513 |
+-----+-------------+----------+----------+
|   8 |   0.119048  | 0.92     | 0.210139 |
+-----+-------------+----------+----------+
|   9 |   0.106032  | 0.921905 | 0.189628 |
+-----+-------------+----------+----------+
|  10 |   0.0954286 | 0.921905 |

## 2.3 multi‑qa‑mpnet‑base‑dot‑v1
---

In [44]:
useracess_exl = UserAccessExl(output_path, "sentence-transformers/multi-qa-mpnet-base-dot-v1")
response = useracess_exl.read_qa_from_excel(path_for_qa)

525 525 525


### 2.3.1 Overlapped Chunks

#### I. Sparse

In [45]:
df_overlapped_sparse = useracess_exl.overlapped_chunking_sparse_embed()
df_overlapped_sparse.to_excel(output_path + 'mpnet_overlapped_chunking_sparse_embed.xlsx')

Success


In [46]:
print("Result: Overlapped Chunking & Sparse Embedding")
show_evaluation(df_overlapped_sparse, chunk_type = 'overlapping')

Result: Overlapped Chunking & Sparse Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.607619 | 0.539048 | 0.561587 |
+-----+-------------+----------+----------+
|   2 |    0.388571 | 0.668095 | 0.481079 |
+-----+-------------+----------+----------+
|   3 |    0.283175 | 0.721175 | 0.39829  |
+-----+-------------+----------+----------+
|   4 |    0.223333 | 0.757841 | 0.338555 |
+-----+-------------+----------+----------+
|   5 |    0.183619 | 0.775683 | 0.291722 |
+-----+-------------+----------+----------+


#### II. Dense

In [47]:
df_overlapped_dense = useracess_exl.overlapped_chunking_dense_embed()
df_overlapped_dense.to_excel(output_path + 'mpnet_overlapped_chunking_dense_embed.xlsx')


Success


In [48]:
print("Result: Overlapped Chunking & Dense Embedding")
show_evaluation(df_overlapped_dense, chunk_type = 'overlapping')

Result: Overlapped Chunking & Dense Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.546667 | 0.480381 | 0.501905 |
+-----+-------------+----------+----------+
|   2 |    0.37619  | 0.63473  | 0.461342 |
+-----+-------------+----------+----------+
|   3 |    0.291429 | 0.730825 | 0.40737  |
+-----+-------------+----------+----------+
|   4 |    0.234286 | 0.781048 | 0.352972 |
+-----+-------------+----------+----------+
|   5 |    0.195429 | 0.814857 | 0.309332 |
+-----+-------------+----------+----------+


#### III. Hybrid

In [49]:
df_overlapped_hyrid = useracess_exl.overlapped_chunking_hyrid_embed()
df_overlapped_hyrid.to_excel(output_path + 'mpnet_overlapped_chunking_hybrid_embed.xlsx')

Success


In [50]:
print("Result: Overlapped Chunking & Hybrind Embedding")
show_evaluation(df_overlapped_dense, chunk_type = 'overlapping', k = 10)

Result: Overlapped Chunking & Hybrind Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |   0.546667  | 0.480381 | 0.501905 |
+-----+-------------+----------+----------+
|   2 |   0.37619   | 0.63473  | 0.461342 |
+-----+-------------+----------+----------+
|   3 |   0.291429  | 0.730825 | 0.40737  |
+-----+-------------+----------+----------+
|   4 |   0.234286  | 0.781048 | 0.352972 |
+-----+-------------+----------+----------+
|   5 |   0.195429  | 0.814857 | 0.309332 |
+-----+-------------+----------+----------+
|   6 |   0.162857  | 0.814857 | 0.266841 |
+-----+-------------+----------+----------+
|   7 |   0.139592  | 0.814857 | 0.23465  |
+-----+-------------+----------+----------+
|   8 |   0.122143  | 0.814857 | 0.209411 |
+-----+-------------+----------+----------+
|   9 |   0.108571  | 0.814857 | 0.189086 |
+-----+-------------+----------+----------+
|  10 |   0.0977143 | 0.8148

### 2.3.2 Recursive Chunking

#### I. Sparse



In [51]:
df_recursive_sparse = useracess_exl.recursive_chunking_sparse_embed()
df_recursive_sparse.to_excel(output_path + 'mpnet_recursive_chunking_sparse_embed.xlsx')


Success


In [52]:
print("Result: Recursive Chunking & Sparse Embedding")
show_evaluation(df_recursive_sparse, chunk_type = 'recursive', k = 5)

Result: Recursive Chunking & Sparse Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.641905 | 0.633333 | 0.63619  |
+-----+-------------+----------+----------+
|   2 |    0.381905 | 0.749524 | 0.504444 |
+-----+-------------+----------+----------+
|   3 |    0.274286 | 0.803492 | 0.407492 |
+-----+-------------+----------+----------+
|   4 |    0.212381 | 0.829206 | 0.337052 |
+-----+-------------+----------+----------+
|   5 |    0.174857 | 0.851746 | 0.289206 |
+-----+-------------+----------+----------+


#### II. Dense

In [53]:
df_recursive_dense = useracess_exl.recursive_chunking_dense_embed()
df_recursive_dense.to_excel(output_path + 'mpnet_recursive_chunking_dense_embed.xlsx')

Success


In [54]:
print("Result: Recursive Chunking & Dense Embedding")
show_evaluation(df_recursive_dense, chunk_type = 'recursive', k = 5)

Result: Recursive Chunking & Dense Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.497143 | 0.484762 | 0.488889 |
+-----+-------------+----------+----------+
|   2 |    0.341905 | 0.667619 | 0.450476 |
+-----+-------------+----------+----------+
|   3 |    0.260952 | 0.76254  | 0.387302 |
+-----+-------------+----------+----------+
|   4 |    0.209048 | 0.812698 | 0.331247 |
+-----+-------------+----------+----------+
|   5 |    0.172952 | 0.840317 | 0.28585  |
+-----+-------------+----------+----------+


#### III. Hybrid

In [55]:
df_recursive_hyrid = useracess_exl.recursive_chunking_hyrid_embed()
df_recursive_hyrid.to_excel(output_path + 'mpnet_recursive_chunking_hyrid_embed.xlsx')


Success


In [56]:
print("Result: Recursive Chunking & Hybrid Embedding")
show_evaluation(df_recursive_hyrid, chunk_type = 'recursive', k = 10)

Result: Recursive Chunking & Hybrid Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |   0.615238  | 0.603492 | 0.607302 |
+-----+-------------+----------+----------+
|   2 |   0.330476  | 0.647302 | 0.436    |
+-----+-------------+----------+----------+
|   3 |   0.221587  | 0.650159 | 0.329397 |
+-----+-------------+----------+----------+
|   4 |   0.168095  | 0.657778 | 0.266957 |
+-----+-------------+----------+----------+
|   5 |   0.136     | 0.665397 | 0.225238 |
+-----+-------------+----------+----------+
|   6 |   0.148254  | 0.872063 | 0.252872 |
+-----+-------------+----------+----------+
|   7 |   0.130612  | 0.894921 | 0.227471 |
+-----+-------------+----------+----------+
|   8 |   0.117143  | 0.914921 | 0.207246 |
+-----+-------------+----------+----------+
|   9 |   0.10455   | 0.91873  | 0.187365 |
+-----+-------------+----------+----------+
|  10 |   0.0940952 | 0.91873 

### 2.3.3 Semantic Chunking

#### I. Sparse

In [57]:
df_semantic_sparse = useracess_exl.semantic_chunking_sparse_embed()
df_semantic_sparse.to_excel(output_path + 'mpnet_semantic_chunking_sparse_embed.xlsx')

Success


In [58]:
print("Result: Semantic Chunking & Sparse Embedding")
show_evaluation(df_semantic_sparse, chunk_type = 'semantic', k = 5)

Result: Semantic Chunking & Sparse Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.619048 | 0.604444 | 0.609206 |
+-----+-------------+----------+----------+
|   2 |    0.378095 | 0.736508 | 0.497397 |
+-----+-------------+----------+----------+
|   3 |    0.272381 | 0.790794 | 0.403175 |
+-----+-------------+----------+----------+
|   4 |    0.214286 | 0.830794 | 0.339247 |
+-----+-------------+----------+----------+
|   5 |    0.174857 | 0.844762 | 0.288503 |
+-----+-------------+----------+----------+


### II. Dense

In [59]:
df_semantic_dense = useracess_exl.semantic_chunking_dense_embed()
df_semantic_dense.to_excel(output_path + 'mpnet_semantic_chunking_dense_embed.xlsx')

Success


In [60]:
print("Result: Semantic Chunking & Dense Embedding")
show_evaluation(df_semantic_dense, chunk_type = 'semantic', k = 5)

Result: Semantic Chunking & Dense Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.470476 | 0.453968 | 0.459365 |
+-----+-------------+----------+----------+
|   2 |    0.319048 | 0.609841 | 0.41581  |
+-----+-------------+----------+----------+
|   3 |    0.247619 | 0.709524 | 0.364571 |
+-----+-------------+----------+----------+
|   4 |    0.202381 | 0.772381 | 0.318712 |
+-----+-------------+----------+----------+
|   5 |    0.168381 | 0.802857 | 0.276803 |
+-----+-------------+----------+----------+


#### III. Hybrid

In [61]:
df_semantic_hybrid = useracess_exl.semantic_chunking_hyrid_embed()
df_semantic_hybrid.to_excel(output_path + 'mpnet_semantic_chunking_hybrid_embed.xlsx')

Success


In [62]:
print("Result: Semantic Chunking & Dense Embedding")
show_evaluation(df_semantic_hybrid, chunk_type = 'semantic', k = 10)

Result: Semantic Chunking & Dense Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |   0.567619  | 0.550159 | 0.555873 |
+-----+-------------+----------+----------+
|   2 |   0.304762  | 0.591111 | 0.400127 |
+-----+-------------+----------+----------+
|   3 |   0.206349  | 0.599365 | 0.30546  |
+-----+-------------+----------+----------+
|   4 |   0.154762  | 0.599365 | 0.244898 |
+-----+-------------+----------+----------+
|   5 |   0.124571  | 0.602222 | 0.205578 |
+-----+-------------+----------+----------+
|   6 |   0.140317  | 0.811429 | 0.238277 |
+-----+-------------+----------+----------+
|   7 |   0.127075  | 0.857143 | 0.220508 |
+-----+-------------+----------+----------+
|   8 |   0.114762  | 0.884762 | 0.202478 |
+-----+-------------+----------+----------+
|   9 |   0.103492  | 0.898095 | 0.185022 |
+-----+-------------+----------+----------+
|  10 |   0.0931429 | 0.898095 |

## 2.4 sentence-transformers/all-mpnet-base-v2

---



In [63]:
useracess_exl = UserAccessExl(output_path, "sentence-transformers/all-mpnet-base-v2")
response = useracess_exl.read_qa_from_excel(path_for_qa)

525 525 525


2.4.1 Overlapped Chunks

In [64]:
df_overlapped_sparse = useracess_exl.overlapped_chunking_sparse_embed()
df_overlapped_sparse.to_excel(output_path + 'mpnet_base_overlapped_chunking_sparse_embed.xlsx')


Success


In [65]:
print("Result: Overlapped Chunking & Sparse Embedding")
show_evaluation(df_overlapped_sparse, chunk_type = 'overlapping')

Result: Overlapped Chunking & Sparse Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.607619 | 0.539048 | 0.561587 |
+-----+-------------+----------+----------+
|   2 |    0.388571 | 0.668095 | 0.481079 |
+-----+-------------+----------+----------+
|   3 |    0.283175 | 0.721175 | 0.39829  |
+-----+-------------+----------+----------+
|   4 |    0.223333 | 0.757841 | 0.338555 |
+-----+-------------+----------+----------+
|   5 |    0.183619 | 0.775683 | 0.291722 |
+-----+-------------+----------+----------+


#### II. Dense

In [66]:
df_overlapped_dense = useracess_exl.overlapped_chunking_dense_embed()
df_overlapped_dense.to_excel(output_path + 'mpnet_base_overlapped_chunking_dense_embed.xlsx')


Success


In [67]:
print("Result: Overlapped Chunking & Dense Embedding")
show_evaluation(df_overlapped_dense, chunk_type = 'overlapping')

Result: Overlapped Chunking & Dense Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.546667 | 0.47419  | 0.497587 |
+-----+-------------+----------+----------+
|   2 |    0.371429 | 0.628857 | 0.456136 |
+-----+-------------+----------+----------+
|   3 |    0.285079 | 0.723841 | 0.400195 |
+-----+-------------+----------+----------+
|   4 |    0.23     | 0.772952 | 0.347226 |
+-----+-------------+----------+----------+
|   5 |    0.192762 | 0.806921 | 0.305364 |
+-----+-------------+----------+----------+


#### III. Hybrid **bold text**

In [68]:
df_overlapped_hyrid = useracess_exl.overlapped_chunking_hyrid_embed()
df_overlapped_hyrid.to_excel(output_path + 'mpnet_base_overlapped_chunking_hybrid_embed.xlsx')

Success


In [69]:
print("Result: Overlapped Chunking & Hybrind Embedding")
show_evaluation(df_overlapped_dense, chunk_type = 'overlapping', k = 10)

Result: Overlapped Chunking & Hybrind Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.546667 | 0.47419  | 0.497587 |
+-----+-------------+----------+----------+
|   2 |    0.371429 | 0.628857 | 0.456136 |
+-----+-------------+----------+----------+
|   3 |    0.285079 | 0.723841 | 0.400195 |
+-----+-------------+----------+----------+
|   4 |    0.23     | 0.772952 | 0.347226 |
+-----+-------------+----------+----------+
|   5 |    0.192762 | 0.806921 | 0.305364 |
+-----+-------------+----------+----------+
|   6 |    0.160635 | 0.806921 | 0.263387 |
+-----+-------------+----------+----------+
|   7 |    0.137687 | 0.806921 | 0.231592 |
+-----+-------------+----------+----------+
|   8 |    0.120476 | 0.806921 | 0.206667 |
+-----+-------------+----------+----------+
|   9 |    0.10709  | 0.806921 | 0.186598 |
+-----+-------------+----------+----------+
|  10 |    0.096381 | 0.8069

### 2.4.2 Recursive Chunking


#### I. Sparse



In [70]:
df_recursive_sparse = useracess_exl.recursive_chunking_sparse_embed()
df_recursive_sparse.to_excel(output_path + 'mpnet_base_recursive_chunking_sparse_embed.xlsx')


Success


In [71]:
print("Result: Recursive Chunking & Sparse Embedding")
show_evaluation(df_recursive_sparse, chunk_type = 'recursive', k = 5)

Result: Recursive Chunking & Sparse Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.641905 | 0.633333 | 0.63619  |
+-----+-------------+----------+----------+
|   2 |    0.381905 | 0.749524 | 0.504444 |
+-----+-------------+----------+----------+
|   3 |    0.274286 | 0.803492 | 0.407492 |
+-----+-------------+----------+----------+
|   4 |    0.212381 | 0.829206 | 0.337052 |
+-----+-------------+----------+----------+
|   5 |    0.174857 | 0.851746 | 0.289206 |
+-----+-------------+----------+----------+


#### II. Dense

In [72]:
df_recursive_dense = useracess_exl.recursive_chunking_dense_embed()
df_recursive_dense.to_excel(output_path + 'mpnet_base_recursive_chunking_dense_embed.xlsx')

Success


In [73]:
print("Result: Recursive Chunking & Dense Embedding")
show_evaluation(df_recursive_dense, chunk_type = 'recursive', k = 5)

Result: Recursive Chunking & Dense Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.56381  | 0.550159 | 0.554603 |
+-----+-------------+----------+----------+
|   2 |    0.371429 | 0.724127 | 0.488825 |
+-----+-------------+----------+----------+
|   3 |    0.272381 | 0.794603 | 0.403937 |
+-----+-------------+----------+----------+
|   4 |    0.21619  | 0.84127  | 0.342676 |
+-----+-------------+----------+----------+
|   5 |    0.176381 | 0.858413 | 0.291655 |
+-----+-------------+----------+----------+


#### III. Hybrid

In [74]:
df_recursive_hyrid = useracess_exl.recursive_chunking_hyrid_embed()
df_recursive_hyrid.to_excel(output_path + 'mpnet_base_recursive_chunking_hyrid_embed.xlsx')


Success


In [75]:
print("Result: Recursive Chunking & Hybrid Embedding")
show_evaluation(df_recursive_hyrid, chunk_type = 'recursive', k = 10)

Result: Recursive Chunking & Hybrid Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.620952 | 0.606349 | 0.611111 |
+-----+-------------+----------+----------+
|   2 |    0.341905 | 0.667302 | 0.450286 |
+-----+-------------+----------+----------+
|   3 |    0.231111 | 0.676825 | 0.343302 |
+-----+-------------+----------+----------+
|   4 |    0.174762 | 0.681587 | 0.277243 |
+-----+-------------+----------+----------+
|   5 |    0.141333 | 0.689206 | 0.233855 |
+-----+-------------+----------+----------+
|   6 |    0.144444 | 0.847302 | 0.246206 |
+-----+-------------+----------+----------+
|   7 |    0.127619 | 0.873016 | 0.22218  |
+-----+-------------+----------+----------+
|   8 |    0.115    | 0.896508 | 0.203359 |
+-----+-------------+----------+----------+
|   9 |    0.102222 | 0.896508 | 0.183111 |
+-----+-------------+----------+----------+
|  10 |    0.092    | 0.896508

### 2.4.3 Semantic Chunking

#### I. Sparse

In [76]:
df_semantic_sparse = useracess_exl.semantic_chunking_sparse_embed()
df_semantic_sparse.to_excel(output_path + 'mpnet_base_semantic_chunking_sparse_embed.xlsx')

Success


In [77]:
print("Result: Semantic Chunking & Sparse Embedding")
show_evaluation(df_semantic_sparse, chunk_type = 'semantic', k = 5)

Result: Semantic Chunking & Sparse Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.619048 | 0.604444 | 0.609206 |
+-----+-------------+----------+----------+
|   2 |    0.378095 | 0.736508 | 0.497397 |
+-----+-------------+----------+----------+
|   3 |    0.272381 | 0.790794 | 0.403175 |
+-----+-------------+----------+----------+
|   4 |    0.214286 | 0.830794 | 0.339247 |
+-----+-------------+----------+----------+
|   5 |    0.174857 | 0.844762 | 0.288503 |
+-----+-------------+----------+----------+


### II. Dense

In [78]:
df_semantic_dense = useracess_exl.semantic_chunking_dense_embed()
df_semantic_dense.to_excel(output_path + 'mpnet_base_semantic_chunking_dense_embed.xlsx')

Success


In [79]:
print("Result: Semantic Chunking & Dense Embedding")
show_evaluation(df_semantic_dense, chunk_type = 'semantic', k = 5)

Result: Semantic Chunking & Dense Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.527619 | 0.509206 | 0.515238 |
+-----+-------------+----------+----------+
|   2 |    0.345714 | 0.663175 | 0.451365 |
+-----+-------------+----------+----------+
|   3 |    0.257143 | 0.738095 | 0.378857 |
+-----+-------------+----------+----------+
|   4 |    0.201905 | 0.774286 | 0.318458 |
+-----+-------------+----------+----------+
|   5 |    0.170286 | 0.817143 | 0.280431 |
+-----+-------------+----------+----------+


#### III. Hybrid

In [80]:
df_semantic_hybrid = useracess_exl.semantic_chunking_hyrid_embed()
df_semantic_hybrid.to_excel(output_path + 'mpnet_base_semantic_chunking_hybrid_embed.xlsx')

Success


In [81]:
print("Result: Semantic Chunking & Dense Embedding")
show_evaluation(df_semantic_hybrid, chunk_type = 'semantic', k = 10)

Result: Semantic Chunking & Dense Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |   0.571429  | 0.553968 | 0.559683 |
+-----+-------------+----------+----------+
|   2 |   0.309524  | 0.59746  | 0.405333 |
+-----+-------------+----------+----------+
|   3 |   0.209524  | 0.605079 | 0.30946  |
+-----+-------------+----------+----------+
|   4 |   0.158095  | 0.608889 | 0.249723 |
+-----+-------------+----------+----------+
|   5 |   0.127619  | 0.613651 | 0.210295 |
+-----+-------------+----------+----------+
|   6 |   0.137778  | 0.79619  | 0.233923 |
+-----+-------------+----------+----------+
|   7 |   0.126531  | 0.853333 | 0.219556 |
+-----+-------------+----------+----------+
|   8 |   0.114524  | 0.882857 | 0.202055 |
+-----+-------------+----------+----------+
|   9 |   0.103069  | 0.894286 | 0.18426  |
+-----+-------------+----------+----------+
|  10 |   0.0927619 | 0.894286 |

## 2.5 sentence-transformers/all-MiniLM-L12-v2

---



In [82]:
useracess_exl = UserAccessExl(output_path, "sentence-transformers/all-MiniLM-L12-v2")
response = useracess_exl.read_qa_from_excel(path_for_qa)

525 525 525


### 2.5.1 Overlapped Chunks

In [83]:
df_overlapped_sparse = useracess_exl.overlapped_chunking_sparse_embed()
df_overlapped_sparse.to_excel(output_path + 'MiniLM_overlapped_chunking_sparse_embed.xlsx')


Success


In [84]:
print("Result: Overlapped Chunking & Sparse Embedding")
show_evaluation(df_overlapped_sparse, chunk_type = 'overlapping')

Result: Overlapped Chunking & Sparse Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.607619 | 0.539048 | 0.561587 |
+-----+-------------+----------+----------+
|   2 |    0.388571 | 0.668095 | 0.481079 |
+-----+-------------+----------+----------+
|   3 |    0.283175 | 0.721175 | 0.39829  |
+-----+-------------+----------+----------+
|   4 |    0.223333 | 0.757841 | 0.338555 |
+-----+-------------+----------+----------+
|   5 |    0.183619 | 0.775683 | 0.291722 |
+-----+-------------+----------+----------+


#### II. Dense

In [85]:
df_overlapped_dense = useracess_exl.overlapped_chunking_dense_embed()
df_overlapped_dense.to_excel(output_path + 'MiniLM_overlapped_chunking_dense_embed.xlsx')


Success


In [86]:
print("Result: Overlapped Chunking & Dense Embedding")
show_evaluation(df_overlapped_dense, chunk_type = 'overlapping')

Result: Overlapped Chunking & Dense Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.527619 | 0.459905 | 0.481714 |
+-----+-------------+----------+----------+
|   2 |    0.349524 | 0.601079 | 0.4322   |
+-----+-------------+----------+----------+
|   3 |    0.254603 | 0.653206 | 0.35858  |
+-----+-------------+----------+----------+
|   4 |    0.202381 | 0.690825 | 0.306952 |
+-----+-------------+----------+----------+
|   5 |    0.17181  | 0.73146  | 0.273342 |
+-----+-------------+----------+----------+


#### III. Hybrid **bold text**

In [87]:
df_overlapped_hyrid = useracess_exl.overlapped_chunking_hyrid_embed()
df_overlapped_hyrid.to_excel(output_path + 'MiniLM_overlapped_chunking_hybrid_embed.xlsx')

Success


In [88]:
print("Result: Overlapped Chunking & Hybrind Embedding")
show_evaluation(df_overlapped_dense, chunk_type = 'overlapping', k = 10)

Result: Overlapped Chunking & Hybrind Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |   0.527619  | 0.459905 | 0.481714 |
+-----+-------------+----------+----------+
|   2 |   0.349524  | 0.601079 | 0.4322   |
+-----+-------------+----------+----------+
|   3 |   0.254603  | 0.653206 | 0.35858  |
+-----+-------------+----------+----------+
|   4 |   0.202381  | 0.690825 | 0.306952 |
+-----+-------------+----------+----------+
|   5 |   0.17181   | 0.73146  | 0.273342 |
+-----+-------------+----------+----------+
|   6 |   0.143175  | 0.73146  | 0.235636 |
+-----+-------------+----------+----------+
|   7 |   0.122721  | 0.73146  | 0.207102 |
+-----+-------------+----------+----------+
|   8 |   0.107381  | 0.73146  | 0.18475  |
+-----+-------------+----------+----------+
|   9 |   0.0954497 | 0.73146  | 0.166763 |
+-----+-------------+----------+----------+
|  10 |   0.0859048 | 0.7314

### 2.5.2 Recursive Chunking


#### I. Sparse



In [89]:
df_recursive_sparse = useracess_exl.recursive_chunking_sparse_embed()
df_recursive_sparse.to_excel(output_path + 'MiniLM_recursive_chunking_sparse_embed.xlsx')


Success


In [90]:
print("Result: Recursive Chunking & Sparse Embedding")
show_evaluation(df_recursive_sparse, chunk_type = 'recursive', k = 5)

Result: Recursive Chunking & Sparse Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.641905 | 0.633333 | 0.63619  |
+-----+-------------+----------+----------+
|   2 |    0.381905 | 0.749524 | 0.504444 |
+-----+-------------+----------+----------+
|   3 |    0.274286 | 0.803492 | 0.407492 |
+-----+-------------+----------+----------+
|   4 |    0.212381 | 0.829206 | 0.337052 |
+-----+-------------+----------+----------+
|   5 |    0.174857 | 0.851746 | 0.289206 |
+-----+-------------+----------+----------+


#### II. Dense

In [91]:
df_recursive_dense = useracess_exl.recursive_chunking_dense_embed()
df_recursive_dense.to_excel(output_path + 'MiniLM_recursive_chunking_dense_embed.xlsx')

Success


In [92]:
print("Result: Recursive Chunking & Dense Embedding")
show_evaluation(df_recursive_dense, chunk_type = 'recursive', k = 5)

Result: Recursive Chunking & Dense Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.504762 | 0.493333 | 0.497143 |
+-----+-------------+----------+----------+
|   2 |    0.305714 | 0.597143 | 0.402857 |
+-----+-------------+----------+----------+
|   3 |    0.233651 | 0.681587 | 0.34654  |
+-----+-------------+----------+----------+
|   4 |    0.188095 | 0.729841 | 0.29785  |
+-----+-------------+----------+----------+
|   5 |    0.156952 | 0.760952 | 0.259206 |
+-----+-------------+----------+----------+


#### III. Hybrid

In [93]:
df_recursive_hyrid = useracess_exl.recursive_chunking_hyrid_embed()
df_recursive_hyrid.to_excel(output_path + 'MiniLM_recursive_chunking_hyrid_embed.xlsx')


Success


In [94]:
print("Result: Recursive Chunking & Hybrid Embedding")
show_evaluation(df_recursive_hyrid, chunk_type = 'recursive', k = 10)

Result: Recursive Chunking & Hybrid Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |   0.552381  | 0.540952 | 0.544762 |
+-----+-------------+----------+----------+
|   2 |   0.309524  | 0.604444 | 0.407746 |
+-----+-------------+----------+----------+
|   3 |   0.208889  | 0.612063 | 0.310349 |
+-----+-------------+----------+----------+
|   4 |   0.158571  | 0.619683 | 0.251719 |
+-----+-------------+----------+----------+
|   5 |   0.128381  | 0.625397 | 0.212358 |
+-----+-------------+----------+----------+
|   6 |   0.149524  | 0.876825 | 0.254845 |
+-----+-------------+----------+----------+
|   7 |   0.133878  | 0.913968 | 0.232974 |
+-----+-------------+----------+----------+
|   8 |   0.118571  | 0.924127 | 0.209666 |
+-----+-------------+----------+----------+
|   9 |   0.105608  | 0.926032 | 0.189172 |
+-----+-------------+----------+----------+
|  10 |   0.0950476 | 0.926032

### 2.5.3 Semantic Chunking

#### I. Sparse

In [95]:
df_semantic_sparse = useracess_exl.semantic_chunking_sparse_embed()
df_semantic_sparse.to_excel(output_path + 'MiniLM_semantic_chunking_sparse_embed.xlsx')

Success


In [96]:
print("Result: Semantic Chunking & Sparse Embedding")
show_evaluation(df_semantic_sparse, chunk_type = 'semantic', k = 5)

Result: Semantic Chunking & Sparse Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.619048 | 0.604444 | 0.609206 |
+-----+-------------+----------+----------+
|   2 |    0.378095 | 0.736508 | 0.497397 |
+-----+-------------+----------+----------+
|   3 |    0.272381 | 0.790794 | 0.403175 |
+-----+-------------+----------+----------+
|   4 |    0.214286 | 0.830794 | 0.339247 |
+-----+-------------+----------+----------+
|   5 |    0.174857 | 0.844762 | 0.288503 |
+-----+-------------+----------+----------+


### II. Dense

In [97]:
df_semantic_dense = useracess_exl.semantic_chunking_dense_embed()
df_semantic_dense.to_excel(output_path + 'MiniLM_semantic_chunking_dense_embed.xlsx')

Success


In [98]:
print("Result: Semantic Chunking & Dense Embedding")
show_evaluation(df_semantic_dense, chunk_type = 'semantic', k = 5)

Result: Semantic Chunking & Dense Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.478095 | 0.461587 | 0.466984 |
+-----+-------------+----------+----------+
|   2 |    0.300952 | 0.576508 | 0.392635 |
+-----+-------------+----------+----------+
|   3 |    0.225397 | 0.649841 | 0.332698 |
+-----+-------------+----------+----------+
|   4 |    0.184286 | 0.707937 | 0.290866 |
+-----+-------------+----------+----------+
|   5 |    0.152381 | 0.730794 | 0.25093  |
+-----+-------------+----------+----------+


#### III. Hybrid

In [99]:
df_semantic_hybrid = useracess_exl.semantic_chunking_hyrid_embed()
df_semantic_hybrid.to_excel(output_path + 'MiniLM_semantic_chunking_hybrid_embed.xlsx')

Success


In [100]:
print("Result: Semantic Chunking & Dense Embedding")
show_evaluation(df_semantic_hybrid, chunk_type = 'semantic', k = 10)

Result: Semantic Chunking & Dense Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |   0.504762  | 0.488254 | 0.493651 |
+-----+-------------+----------+----------+
|   2 |   0.28      | 0.539365 | 0.366286 |
+-----+-------------+----------+----------+
|   3 |   0.188571  | 0.545079 | 0.278603 |
+-----+-------------+----------+----------+
|   4 |   0.142381  | 0.547937 | 0.224834 |
+-----+-------------+----------+----------+
|   5 |   0.115048  | 0.553651 | 0.189615 |
+-----+-------------+----------+----------+
|   6 |   0.138095  | 0.8      | 0.234603 |
+-----+-------------+----------+----------+
|   7 |   0.126803  | 0.85619  | 0.220085 |
+-----+-------------+----------+----------+
|   8 |   0.115476  | 0.891429 | 0.20379  |
+-----+-------------+----------+----------+
|   9 |   0.104127  | 0.904762 | 0.186199 |
+-----+-------------+----------+----------+
|  10 |   0.0937143 | 0.904762 |

## 2.6 flax-sentence-embeddings/all_datasets_v3_MiniLM-L12

In [5]:
useracess_exl = UserAccessExl(output_path, "flax-sentence-embeddings/all_datasets_v3_MiniLM-L12")
response = useracess_exl.read_qa_from_excel(path_for_qa)

525 525 525


### 2.6.1 Overlapped Chunks

In [8]:
df_overlapped_sparse = useracess_exl.overlapped_chunking_sparse_embed()
df_overlapped_sparse.to_excel(output_path + 'v3_overlapped_chunking_sparse_embed.xlsx')


Success


In [11]:
print("Result: Overlapped Chunking & Sparse Embedding")
show_evaluation(df_overlapped_sparse, chunk_type = 'overlapping')

Result: Overlapped Chunking & Sparse Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.607619 | 0.539048 | 0.561587 |
+-----+-------------+----------+----------+
|   2 |    0.388571 | 0.668095 | 0.481079 |
+-----+-------------+----------+----------+
|   3 |    0.283175 | 0.721175 | 0.39829  |
+-----+-------------+----------+----------+
|   4 |    0.223333 | 0.757841 | 0.338555 |
+-----+-------------+----------+----------+
|   5 |    0.183619 | 0.775683 | 0.291722 |
+-----+-------------+----------+----------+


#### II. Dense

In [12]:
df_overlapped_dense = useracess_exl.overlapped_chunking_dense_embed()
df_overlapped_dense.to_excel(output_path + 'v3_overlapped_chunking_dense_embed.xlsx')


Success


In [13]:
print("Result: Overlapped Chunking & Dense Embedding")
show_evaluation(df_overlapped_dense, chunk_type = 'overlapping')

Result: Overlapped Chunking & Dense Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.558095 | 0.492921 | 0.513778 |
+-----+-------------+----------+----------+
|   2 |    0.364762 | 0.625524 | 0.450422 |
+-----+-------------+----------+----------+
|   3 |    0.274286 | 0.697651 | 0.38512  |
+-----+-------------+----------+----------+
|   4 |    0.216667 | 0.73146  | 0.32756  |
+-----+-------------+----------+----------+
|   5 |    0.180571 | 0.761619 | 0.286608 |
+-----+-------------+----------+----------+


#### III. Hybrid **bold text**

In [14]:
df_overlapped_hyrid = useracess_exl.overlapped_chunking_hyrid_embed()
df_overlapped_hyrid.to_excel(output_path + 'v3_overlapped_chunking_hybrid_embed.xlsx')

Success


In [15]:
print("Result: Overlapped Chunking & Hybrind Embedding")
show_evaluation(df_overlapped_dense, chunk_type = 'overlapping', k = 10)

Result: Overlapped Chunking & Hybrind Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |   0.558095  | 0.492921 | 0.513778 |
+-----+-------------+----------+----------+
|   2 |   0.364762  | 0.625524 | 0.450422 |
+-----+-------------+----------+----------+
|   3 |   0.274286  | 0.697651 | 0.38512  |
+-----+-------------+----------+----------+
|   4 |   0.216667  | 0.73146  | 0.32756  |
+-----+-------------+----------+----------+
|   5 |   0.180571  | 0.761619 | 0.286608 |
+-----+-------------+----------+----------+
|   6 |   0.150476  | 0.761619 | 0.247148 |
+-----+-------------+----------+----------+
|   7 |   0.12898   | 0.761619 | 0.217272 |
+-----+-------------+----------+----------+
|   8 |   0.112857  | 0.761619 | 0.193858 |
+-----+-------------+----------+----------+
|   9 |   0.100317  | 0.761619 | 0.175011 |
+-----+-------------+----------+----------+
|  10 |   0.0902857 | 0.7616

### 2.6.2 Recursive Chunking





#### I. Sparse



In [16]:
df_recursive_sparse = useracess_exl.recursive_chunking_sparse_embed()
df_recursive_sparse.to_excel(output_path + 'v3_recursive_chunking_sparse_embed.xlsx')


Success


In [17]:
print("Result: Recursive Chunking & Sparse Embedding")
show_evaluation(df_recursive_sparse, chunk_type = 'recursive', k = 5)

Result: Recursive Chunking & Sparse Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.641905 | 0.633333 | 0.63619  |
+-----+-------------+----------+----------+
|   2 |    0.381905 | 0.749524 | 0.504444 |
+-----+-------------+----------+----------+
|   3 |    0.274286 | 0.803492 | 0.407492 |
+-----+-------------+----------+----------+
|   4 |    0.212381 | 0.829206 | 0.337052 |
+-----+-------------+----------+----------+
|   5 |    0.174857 | 0.851746 | 0.289206 |
+-----+-------------+----------+----------+


#### II. Dense

In [18]:
df_recursive_dense = useracess_exl.recursive_chunking_dense_embed()
df_recursive_dense.to_excel(output_path + 'v3_recursive_chunking_dense_embed.xlsx')

Success


In [19]:
print("Result: Recursive Chunking & Dense Embedding")
show_evaluation(df_recursive_dense, chunk_type = 'recursive', k = 5)

Result: Recursive Chunking & Dense Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.506667 | 0.493968 | 0.498095 |
+-----+-------------+----------+----------+
|   2 |    0.322857 | 0.629206 | 0.424889 |
+-----+-------------+----------+----------+
|   3 |    0.24127  | 0.703175 | 0.357651 |
+-----+-------------+----------+----------+
|   4 |    0.190476 | 0.739365 | 0.30166  |
+-----+-------------+----------+----------+
|   5 |    0.161524 | 0.784127 | 0.266893 |
+-----+-------------+----------+----------+


#### III. Hybrid

In [20]:
df_recursive_hyrid = useracess_exl.recursive_chunking_hyrid_embed()
df_recursive_hyrid.to_excel(output_path + 'v3_recursive_chunking_hyrid_embed.xlsx')


Success


In [21]:
print("Result: Recursive Chunking & Hybrid Embedding")
show_evaluation(df_recursive_hyrid, chunk_type = 'recursive', k = 10)

Result: Recursive Chunking & Hybrid Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |   0.56      | 0.546349 | 0.550794 |
+-----+-------------+----------+----------+
|   2 |   0.314286  | 0.613016 | 0.413778 |
+-----+-------------+----------+----------+
|   3 |   0.212063  | 0.620635 | 0.314921 |
+-----+-------------+----------+----------+
|   4 |   0.16      | 0.624444 | 0.253878 |
+-----+-------------+----------+----------+
|   5 |   0.130667  | 0.636825 | 0.216168 |
+-----+-------------+----------+----------+
|   6 |   0.147302  | 0.864444 | 0.251104 |
+-----+-------------+----------+----------+
|   7 |   0.132517  | 0.904444 | 0.230593 |
+-----+-------------+----------+----------+
|   8 |   0.117381  | 0.915873 | 0.207627 |
+-----+-------------+----------+----------+
|   9 |   0.104762  | 0.919683 | 0.187711 |
+-----+-------------+----------+----------+
|  10 |   0.0942857 | 0.919683

### 2.6.3 Semantic Chunking

#### I. Sparse

In [22]:
df_semantic_sparse = useracess_exl.semantic_chunking_sparse_embed()
df_semantic_sparse.to_excel(output_path + 'v3_semantic_chunking_sparse_embed.xlsx')

Success


In [23]:
print("Result: Semantic Chunking & Sparse Embedding")
show_evaluation(df_semantic_sparse, chunk_type = 'semantic', k = 5)

Result: Semantic Chunking & Sparse Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.619048 | 0.604444 | 0.609206 |
+-----+-------------+----------+----------+
|   2 |    0.378095 | 0.736508 | 0.497397 |
+-----+-------------+----------+----------+
|   3 |    0.272381 | 0.790794 | 0.403175 |
+-----+-------------+----------+----------+
|   4 |    0.214286 | 0.830794 | 0.339247 |
+-----+-------------+----------+----------+
|   5 |    0.174857 | 0.844762 | 0.288503 |
+-----+-------------+----------+----------+


### II. Dense

In [24]:
df_semantic_dense = useracess_exl.semantic_chunking_dense_embed()
df_semantic_dense.to_excel(output_path + 'v3_semantic_chunking_dense_embed.xlsx')

Success


In [25]:
print("Result: Semantic Chunking & Dense Embedding")
show_evaluation(df_semantic_dense, chunk_type = 'semantic', k = 5)

Result: Semantic Chunking & Dense Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |    0.462857 | 0.444444 | 0.450476 |
+-----+-------------+----------+----------+
|   2 |    0.306667 | 0.586984 | 0.399937 |
+-----+-------------+----------+----------+
|   3 |    0.232381 | 0.670794 | 0.343175 |
+-----+-------------+----------+----------+
|   4 |    0.184286 | 0.706984 | 0.290739 |
+-----+-------------+----------+----------+
|   5 |    0.15619  | 0.750794 | 0.25737  |
+-----+-------------+----------+----------+


#### III. Hybrid

In [26]:
df_semantic_hybrid = useracess_exl.semantic_chunking_hyrid_embed()
df_semantic_hybrid.to_excel(output_path + 'v3_semantic_chunking_hybrid_embed.xlsx')

Success


In [27]:
print("Result: Semantic Chunking & Dense Embedding")
show_evaluation(df_semantic_hybrid, chunk_type = 'semantic', k = 10)

Result: Semantic Chunking & Dense Embedding
+-----+-------------+----------+----------+
|   K |   Precision |   Recall |       F1 |
+=====+=============+==========+==========+
|   1 |   0.506667  | 0.487302 | 0.493651 |
+-----+-------------+----------+----------+
|   2 |   0.293333  | 0.560317 | 0.382159 |
+-----+-------------+----------+----------+
|   3 |   0.19746   | 0.566032 | 0.290794 |
+-----+-------------+----------+----------+
|   4 |   0.149524  | 0.571746 | 0.235628 |
+-----+-------------+----------+----------+
|   5 |   0.121905  | 0.582222 | 0.200499 |
+-----+-------------+----------+----------+
|   6 |   0.13873   | 0.801905 | 0.235556 |
+-----+-------------+----------+----------+
|   7 |   0.127347  | 0.86     | 0.221037 |
+-----+-------------+----------+----------+
|   8 |   0.115238  | 0.889524 | 0.203367 |
+-----+-------------+----------+----------+
|   9 |   0.10455   | 0.908571 | 0.186961 |
+-----+-------------+----------+----------+
|  10 |   0.0940952 | 0.908571 |

# 3. Generator

In [6]:
import csv
import pandas as pd

aug_obj = Augmentation()
gen_obj = Generator()

class ReadQCA:
    def  __init__(self, filepath):
        self.filepath = filepath
        self.read_excel_content()

    def read_excel_content(self):
        df = pd.read_excel(self.filepath)
        question = df['query']
        retrival = df['retrival']
        self.qc = zip(question, retrival)
    
    def make_llm_call(self, responsepath = output_path  +"/chat/", llm = "llama"):
        count = 0

        csv_file = responsepath + llm  +"_results.csv"

        # Create CSV with header only once
        if not os.path.exists(csv_file):
            with open(csv_file, mode="w", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)
                writer.writerow(["SN", "Question", "Answer"])   # header row
                
        for q, r in self.qc:
            count += 1

            if llm == "llama":
                prompt = aug_obj.prompt_for_llama3(q, r)
                response = gen_obj.chat_with_llama(prompt)
            elif llm == "gpt":
                prompt = aug_obj.prompt_for_gpt_oss_20b(q, r)
                response = gen_obj.chat_with_gpt_oss_20b(prompt)

            elif llm == "gimma":
                if count < 494:
                    continue
                prompt = aug_obj.prompt_for_gimma3_12b(q, r)
                response = gen_obj.chat_with_gimma3_12b(prompt)
        

            # --- Append to CSV ---delimiter = "~|~"
            delimiter = "~|~"
            with open(csv_file, mode="a", encoding="utf-8") as f:
                f.write(f"{count}{delimiter}{q}{delimiter}{response}{delimiter}\n")

            print(f"Success: {count}, response: {response}")



In [9]:
obj = ReadQCA(output_path + 'e5_large_recursive_chunking_dense_embed.xlsx')

In [10]:
obj.make_llm_call(llm="gpt")

Success: 1, response: Answer:  
The general public can obtain the following services directly from Nepal Rastra Bank (at its headquarters or provincial offices):  

1. **Coin and note exchange** – exchanging currency notes and coins.  
2. **Foreign‑currency exchange** – buying or selling foreign currency.  
3. **Purchase of precious metals** – buying gold, silver and gold‑coins (including gold bars of 2.5 g, 5 g and 10 g, and silver bullion/coins).  
4. **Purchase of souvenir coins and medallions** – available through the Mint Division (Babarmahal) and provincial offices.  
5. **Purchase and sale of government bonds** – access to government bond transactions.  
6. **Complaint handling** – lodging and hearing complaints regarding the activities of banks and other financial institutions licensed by the Nepal Rastra Bank.
Success: 2, response: Answer:  
From Nepal Rastra Bank’s provincial offices or its headquarters, the general public can directly:  

- Exchange coins and notes  
- Excha

In [6]:
obj = ReadQCA(output_path + 'e5_large_recursive_chunking_dense_embed.xlsx')
obj.make_llm_call(llm="llama")

Success: 1, response: According to the provided context, the general public can receive the following services directly from Nepal Rastra Bank:

1. Coin and note exchange
2. Foreign currency exchange
3. Purchase of gold, silver, and gold coins
4. Souvenir coins purchase
5. Purchase and sale of government bonds
6. Hearing and addressing any complaints regarding the activities of banks and financial institutions licensed by Nepal Rastra Bank.

Additionally, the general public can also receive services from Nepal Rastra Bank's headquarters, including:

1. Coin and note exchange
2. Foreign currency exchange
3. Purchase of gold, silver, and silver coins
4. Souvenir coins purchase
5. Purchase/sale of government bonds
6. Hearing and addressing any complaints regarding the activities of banks and financial institutions licensed by Nepal Rastra Bank.

It is essential to note that these services are only available at specific branches or locations specified in the context, such as the Currency M

In [ ]:
obj = ReadQCA(output_path + 'e5_large_recursive_chunking_dense_embed.xlsx')
obj.make_llm_call(llm="gimma")

Success: 1, response: The general public can receive services related to coin and note exchange, foreign currency exchange, purchase of gold, silver and gold coins, souvenir coins, purchase and sale of government bonds from both the provincial offices and the headquarters of Nepal Rastra Bank. Additionally, the general public can also have their complaints regarding the activities of banks and financial institutions heard and addressed. Souvenir coins and medallions can be purchased from the Currency Management Department, Minting Division, Babarmahal, and gold and silver bullion/coins can be purchased from the Mint Division under the Currency Management Department, Babarmahal and provincial offices.
Success: 2, response: The general public can receive services related to coin and note exchange, foreign currency exchange, purchase of gold, silver and gold coins, souvenir coins, purchase and sale of government bonds from both the provincial offices and Nepal Rastra Bank Headquarters. Ad

In [ ]:
obj = ReadQCA(output_path + 'e5_large_recursive_chunking_dense_embed.xlsx')
obj.make_llm_call(llm="gimma")

Success: 205, response: Investing in cryptocurrency is risky for several reasons. Firstly, its value is determined purely on market demand and supply, making it highly volatile and susceptible to drastic price drops. Secondly, cryptocurrency is issued by the private sector without a guarantee or security, unlike currencies issued by sovereign countries. Thirdly, rumors of attractive returns can be spread unilaterally, leading to speculative and risky investments. The collapse of exchanges like FTX and cryptocurrencies like TerraUSD and LUNA further demonstrate the inherent risks. Lastly, the Foreign Investment Prohibition Act, 2021 prohibits cryptocurrency investment, potentially leading to the loss of the entire investment amount.

Success: 206, response: Answer: Cryptocurrency can be obtained mainly by mining and purchasing. Mining refers to the act of keeping a record of transactions on a cryptocurrency network, which is also called validation. Another means is to purchase cryptocur

In [7]:
obj = ReadQCA(output_path + 'e5_large_recursive_chunking_dense_embed.xlsx')
obj.make_llm_call(llm="gimma")

Success: 494, response: The y-o-y wholesale price inflation stood at 2.75 percent in mid-September 2024.
Success: 495, response: The y-o-y wholesale price inflation stood at 2.75 percent in mid-September 2024 compared to 4.78 percent a year ago.
Success: 496, response: The y-o-y wholesale price of consumption goods increased by 1.39 percent.
Success: 497, response: The y-o-y wholesale price of intermediate goods increased by 3.56 percent.
Success: 498, response: 53.2 percent
Success: 499, response: 45.6 percent.
Success: 500, response: Capital goods contributed 1.2 percent to total exports in the review period.
Success: 501, response: The ratio of intermediate goods in total exports decreased from 54.1 percent in the same period of the previous year to 53.2 percent in the review period.
Success: 502, response: The ratio of capital goods in total exports was 0.23 percent in the same period of the previous year.
Success: 503, response: 49.7 percent of total imports were intermediate good

In [7]:
from langchain.prompts import PromptTemplate


class InContextAugmentation:
  def __init__(self,):
    pass

  def prompt_for_llama3(self, query, contexts):
    # LLaMA 3 chat-style prompt template
    base_system_prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
    You are a helpful assistant in a retrieval-augmented generation (RAG) system.
    Use the following context to answer the user's question as accurately and informatively as possible.
    If the context is irrelevant, incomplete, or does not contain the answer, reply with:
      "I don’t know based on the given context."
    -Do NOT make up facts or provide external knowledge.
    Do not write any pre-fix befor starting the answer such as 'According to the provided context'; directly start answer after Answer:.
    
    Few-shot-example:  
    Q1: "What are the main functions of Nepal Rastra Bank?"
    A1: "The main functions of Nepal Rastra Bank are issuing Nepalese currency and coins, licensing and regulating banks and financial institutions, formulating monetary and foreign exchange policies, managing foreign exchange reserves, acting as banker and financial advisor to the Government of Nepal, promoting and regulating payment and settlement systems, and managing liquidity." 
    Q2: "How many types of payment cards are prevalent in Nepal?"
    A2: "There are currently three types of payment cards prevalent in Nepal: credit cards, debit cards, and prepaid cards."
    Q3: "Is accounting for investments and loans mandatory? 
    A3: "Yes, accounting for foreign investment and foreign loans is mandatory in Nepal to repatriate unaccounted investment and loan proceeds and to buy or sell shares held in the name of foreign investors.
    Q4: "What was the y-o-y (y-o-y) expansion of M2 in mid-September 2024?"
    A4: "M2 expanded 13.9 percent y-o-y."

    Context:
    {context}

    <|start_header_id|>user<|end_header_id|>
    {query}

    <|start_header_id|>assistant<|end_header_id|>"""

    # Build LangChain PromptTemplate
    prompt_template = PromptTemplate(
        input_variables=["query", "context"],
        template=base_system_prompt
    )

    # Format prompt with retrieved context
    prompt = prompt_template.format(
        query=query,
        context=Augmentation.doc_to_text(contexts)
    )

    return prompt

  def prompt_for_mistral(self):
    pass

  def prompt_for_gimma3_12b(self, query, contexts):
    # RAG-style system prompt template
    base_system_prompt = """<|start|>system<|message|>
    You are a helpful assistant in a retrieval-augmented generation (RAG) system.
    Do not write any pre-fix befor starting the answer such as 'According to the provided context'; directly start answer after Answer: 
    
    Instruction:
    - Use ONLY the provided context to answer the user’s question.
    - If the context is irrelevant, incomplete, or does not contain the answer, reply with:
      "I don’t know based on the given context."
    - Do NOT make up facts or provide external knowledge.

    Few-shot-example:  
    Q1: "What are the main functions of Nepal Rastra Bank?"
    A1: "The main functions of Nepal Rastra Bank are issuing Nepalese currency and coins, licensing and regulating banks and financial institutions, formulating monetary and foreign exchange policies, managing foreign exchange reserves, acting as banker and financial advisor to the Government of Nepal, promoting and regulating payment and settlement systems, and managing liquidity." 
    Q2: "How many types of payment cards are prevalent in Nepal?"
    A2: "There are currently three types of payment cards prevalent in Nepal: credit cards, debit cards, and prepaid cards."
    Q3: "Is accounting for investments and loans mandatory? 
    A3: "Yes, accounting for foreign investment and foreign loans is mandatory in Nepal to repatriate unaccounted investment and loan proceeds and to buy or sell shares held in the name of foreign investors.
    Q4: "What was the y-o-y (y-o-y) expansion of M2 in mid-September 2024?"
    A4: "M2 expanded 13.9 percent y-o-y."

    Context:
    {context}
    <|end|>
    
    <|start|>user<|message|>
    Question: {query}
    <|end|>

    <|start|>assistant<|message|>"""

    # Build LangChain PromptTemplate
    prompt_template = PromptTemplate(
        input_variables=["query", "context"],
        template=base_system_prompt
    )

    # Example input (normally you'd inject retrieved docs here)
    prompt = prompt_template.format(
        query = query,
        context = Augmentation.doc_to_text(contexts)
    )

    return prompt

  def prompt_for_gpt_oss_20b(self, query, contexts):
    base_system_prompt = """<|start|>system<|message|>
    You are a helpful assistant in a retrieval-augmented generation (RAG) system.

    Guidelines:
    - Use ONLY the provided context to answer the user’s question.
    - Do NOT add prefixes like "According to the context" — start directly with the answer.
    - If the context does not contain the answer, reply: "I don’t know based on the given context."
    - Do NOT invent facts or use external knowledge.
    - Keep answers concise, factual, and well-structured.

    Knowledge cutoff: 2024-06
    Current date: {{ currentDate }}
    Reasoning: none
    Web access: disabled
    Tools: disabled
    <|end|>

    <|start|>user<|message|>
    Question: {query}

    Examples:
    Q1: "What are the main functions of Nepal Rastra Bank?"
    A1: "Issuing currency, regulating banks, formulating monetary policy, managing reserves, advising government, regulating payments, managing liquidity."

    Q2: "How many types of payment cards are prevalent in Nepal?"
    A2: "Three types: credit cards, debit cards, and prepaid cards."

    Q3: "Is accounting for investments and loans mandatory?"
    A3: "Yes, accounting for foreign investment and loans is mandatory in Nepal to repatriate unaccounted proceeds and manage share transactions."

    Q4: "What was the y-o-y expansion of M2 in mid-September 2024?"
    A4: "M2 expanded 13.9 percent y-o-y."

    Context:
    {context}
    <|end|>

    <|start|>assistant<|message|>"""

    # Build LangChain PromptTemplate
    prompt_template = PromptTemplate(
        input_variables=["query", "context"],
        template=base_system_prompt
    )

    # Example input (normally you'd inject retrieved docs here)
    prompt = prompt_template.format(
        query = query,
        context = InContextAugmentation.doc_to_text(contexts)
    )

    return prompt

  def doc_to_text(contexts):
    return contexts
  


import csv
import pandas as pd
import os

aug_obj = InContextAugmentation()
gen_obj = Generator()

class ReadQA_With_Examples:
    def  __init__(self, filepath):
        self.filepath = filepath
        self.read_excel_content()

    def read_excel_content(self):
        df = pd.read_excel(self.filepath)
        question = df['query']
        retrival = df['retrival']
        self.qc = zip(question, retrival)
    
    def make_llm_call(self, responsepath = output_path  +"/chat/", llm = "llama"):
        count = 0
        prompt_template_counter = 0

        csv_file = responsepath + llm  +"_in_context_results.csv"

        # Create CSV with header only once
        if not os.path.exists(csv_file):
            with open(csv_file, mode="w", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)
                writer.writerow(["SN", "Question", "Answer"])   # header row
                
        for q, r in self.qc:
            count += 1

            if llm == "llama":
                prompt = aug_obj.prompt_for_llama3(q, r)
                response = gen_obj.chat_with_llama(prompt)
            elif llm == "gpt":
                prompt = aug_obj.prompt_for_gpt_oss_20b(q, r)
                response = gen_obj.chat_with_gpt_oss_20b(prompt)
            elif llm == "gimma":
                if count < 294:
                    continue
                prompt = aug_obj.prompt_for_gimma3_12b(q, r)
                response = gen_obj.chat_with_gimma3_12b(prompt)
        
      
            # --- Append to CSV ---delimiter = "~|~"
            delimiter = "~|~"
            with open(csv_file, mode="a", encoding="utf-8") as f:
                f.write(f"{count}{delimiter}{q}{delimiter}{response}{delimiter}\n")

            print(f"Success: {count}, question:{q},  response: {response}")



In [11]:
obj = ReadQA_With_Examples(output_path + 'e5_large_recursive_chunking_dense_embed.xlsx')
obj.make_llm_call(llm="gpt")

Success: 1, question:What services can the general public receive directly from Nepal Rastra Bank?,  response: The general public can obtain the following services directly from Nepal Rastra Bank (at provincial offices or the headquarters):

- Exchange of coins and banknotes (coin & note exchange)  
- Exchange of foreign currency (foreign‑currency exchange)  
- Purchase of gold and silver bullion/coins, including gold bars of 2.5 g, 5 g, and 10 g, and silver coins (gold/silver bullion purchase)  
- Purchase of souvenir coins and medallions (souvenir‑coin sales)  
- Purchase and sale of government bonds (government‑bond transactions)  
- Filing and hearing of complaints about the activities of banks and other financial institutions licensed by Nepal Rastra Bank.
Success: 2, question:What services can the general public receive directly from Nepal Rastra Bank? ,  response: The general public can obtain the following services directly from Nepal Rastra Bank (at its provincial offices or h

In [ ]:
obj = ReadQA_With_Examples(output_path + 'e5_large_recursive_chunking_dense_embed.xlsx')
obj.make_llm_call(llm="llama")

In [ ]:
obj = ReadQA_With_Examples(output_path + 'e5_large_recursive_chunking_dense_embed.xlsx')
obj.make_llm_call(llm="gimma")

Success: 1, question:What services can the general public receive directly from Nepal Rastra Bank?,  response: From Nepal Rastra Bank, the general public can receive services related to coin and note exchange, foreign currency exchange, purchase of gold, silver and silver coins, souvenir coins, purchase/sale of government bonds, and hearing and addressing any complaints regarding the activities of banks and financial institutions and institutions licensed by Nepal Rastra Bank.
Success: 2, question:What services can the general public receive directly from Nepal Rastra Bank? ,  response: From Nepal Rastra Bank, the general public can receive services related to coin and note exchange, foreign currency exchange, purchase of gold, silver and silver coins, souvenir coins, purchase/sale of government bonds, and hearing and addressing any complaints regarding the activities of banks and financial institutions.
Success: 3, question:How can one identify whether an organization is licensed/perm

In [ ]:
obj = ReadQA_With_Examples(output_path + 'e5_large_recursive_chunking_dense_embed.xlsx')
obj.make_llm_call(llm="gimma")

Success: 124, question:What is a debit card?,  response: A debit card is a card issued in the name of a customer to allow withdrawals and payments from a bank account.
Success: 125, question:What is a credit card?,  response: A credit card is a card that allows the customer to make payments even if they do not have money in their bank account. The cardholder must repay the amount spent on such a card to the relevant bank or financial institution after a certain period of time, along with a specified fee.
Success: 126, question:How many types of payment cards are prevalent in Nepal?,  response: There are currently three types of payment cards prevalent in Nepal: credit card, debit card, and prepaid card.
Success: 127, question:How much is the fee for withdrawing money from an ATM of another bank using an ATM card of one bank?,  response: When a payment card issued by one bank and financial institution is used at an ATM of another bank and financial institution within the country, a fee 

In [8]:
obj = ReadQA_With_Examples(output_path + 'e5_large_recursive_chunking_dense_embed.xlsx')
obj.make_llm_call(llm="gimma")

Success: 294, question:What is the selling rate of 1 Swiss Franc (CHF)?,  response: The selling rate of 1 Swiss Franc (CHF) is 176.14 NPR.
Success: 295, question:What is the buying rate of 1 Australian Dollar (AUD)?,  response: The buying rate of 1 Australian Dollar (AUD) is 92.22 NPR.
Success: 296, question:What is the selling rate of 1 Australian Dollar (AUD)?,  response: The selling rate of 1 Australian Dollar (AUD) is 92.62 NPR.
Success: 297, question:What is the buying rate of 1 Saudi Arabian Riyal (SAR)?,  response: 1 = 37.56NPR(Buy), 37.72 NPR (Sell)
Success: 298, question:What is the buying rate of 1 Qatari Riyal (QAR)?,  response: 1 Qatari Riyal (QAR) has a buying rate of 38.67 NPR.
Success: 299, question:What is the selling rate of 1 Qatari Riyal (QAR)?,  response: 38.83 NPR(Sell)
Success: 300, question: What does RTGS system membership require?,  response: RTGS system membership is open to entities that meet the following criteria: i. Licensed financial institutions regulate

# Evaluating Generator

In [7]:
import pandas as pd

eval_obj = GeneratorEvaluator()

def read_chat_excel():
    directory = output_path + "chat/exl/"
    xlsx_files = [f for f in os.listdir(directory) if f.endswith(".xlsx")]
    for file in xlsx_files:
        print(file)
    
        try:
            file_path = os.path.join(directory, file)
            df = pd.read_excel(file_path)
            sn = df["sn"]
            qa = df["question"]
            answer = df["answer"].fillna("N/A").astype(str)
            reference = df["reference"].fillna("N/A").astype(str)
            print("=============================")
            print(file_path)
            print("=============================")
            rs, bs = eval_generator(answer.tolist(), reference.tolist())
            print("ROUGE Score:", rs)
            print("BERT Score:", bs)
        except Exception as e:
            print(e)
        
def eval_generator(generated, refernce):
    rouge_score = eval_obj.calculate_rouge(generated, refernce)
    bert_score = eval_obj.calculate_bert_score(generated, refernce)
    return rouge_score, bert_score

In [7]:
read_chat_excel()

gimma_in_context_results.xlsx
output/chat/exl/gimma_in_context_results.xlsx


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROUGE Score: {'rouge1': {'precision': 0.6369874710682439, 'recall': 0.7938519994414707, 'f1': 0.6745069094604318}, 'rouge2': {'precision': 0.4944676468510959, 'recall': 0.611390967990908, 'f1': 0.5218921465757874}, 'rougeL': {'precision': 0.5847008922385354, 'recall': 0.7278796494276069, 'f1': 0.619418469667238}}
BERT Score: {'precision': 0.927574634552002, 'recall': 0.9456157088279724, 'f1': 0.9361273050308228}
gimma_results.xlsx
output/chat/exl/gimma_results.xlsx


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROUGE Score: {'rouge1': {'precision': 0.6266314555759909, 'recall': 0.8107653896875779, 'f1': 0.6687772382424819}, 'rouge2': {'precision': 0.4909696037800951, 'recall': 0.6269287601399685, 'f1': 0.5190377644446771}, 'rougeL': {'precision': 0.5752252865024408, 'recall': 0.7435683330393315, 'f1': 0.6138132056791251}}
BERT Score: {'precision': 0.9240190386772156, 'recall': 0.9471586346626282, 'f1': 0.9350123405456543}
gpt_in_context_results.xlsx
output/chat/exl/gpt_in_context_results.xlsx


...

ROUGE Score: {'rouge1': {'precision': 0.5811339246031405, 'recall': 0.7089040733463715, 'f1': 0.5557318329385577}, 'rouge2': {'precision': 0.4391083523696075, 'recall': 0.47188758447904466, 'f1': 0.3948137568922707}, 'rougeL': {'precision': 0.528140939261444, 'recall': 0.6191330576420973, 'f1': 0.4981295322715515}}
BERT Score: {'precision': 0.8897323608398438, 'recall': 0.9216877818107605, 'f1': 0.9047080278396606}
llama_in_context_results.xlsx
output/chat/exl/llama_in_context_results.xlsx


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROUGE Score: {'rouge1': {'precision': 0.5108540475400134, 'recall': 0.8282461078354025, 'f1': 0.5882730096084126}, 'rouge2': {'precision': 0.37974820429727274, 'recall': 0.6024653361333546, 'f1': 0.4335418212332945}, 'rougeL': {'precision': 0.4581774002556515, 'recall': 0.7405444892047938, 'f1': 0.5275838590424106}}
BERT Score: {'precision': 0.9005054831504822, 'recall': 0.9395279288291931, 'f1': 0.9191817045211792}
llama_results.xlsx
output/chat/exl/llama_results.xlsx


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROUGE Score: {'rouge1': {'precision': 0.4965818314325649, 'recall': 0.8441693723201981, 'f1': 0.5810611777108666}, 'rouge2': {'precision': 0.3722644493089244, 'recall': 0.6151298212189595, 'f1': 0.4317980799365111}, 'rougeL': {'precision': 0.44554259919126094, 'recall': 0.7550093042394351, 'f1': 0.5215252137579006}}
BERT Score: {'precision': 0.8979816436767578, 'recall': 0.940869927406311, 'f1': 0.9184987545013428}


In [8]:
read_chat_excel()

gimma_in_context_results.xlsx
output/chat/exl/gimma_in_context_results.xlsx


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROUGE Score: {'rouge1': {'precision': 0.6369874710682439, 'recall': 0.7938519994414707, 'f1': 0.6745069094604318}, 'rouge2': {'precision': 0.4944676468510959, 'recall': 0.611390967990908, 'f1': 0.5218921465757874}, 'rougeL': {'precision': 0.5847008922385354, 'recall': 0.7278796494276069, 'f1': 0.619418469667238}}
BERT Score: {'precision': 0.927574634552002, 'recall': 0.9456157088279724, 'f1': 0.9361273050308228}
gimma_results.xlsx
output/chat/exl/gimma_results.xlsx


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROUGE Score: {'rouge1': {'precision': 0.6266314555759909, 'recall': 0.8107653896875779, 'f1': 0.6687772382424819}, 'rouge2': {'precision': 0.4909696037800951, 'recall': 0.6269287601399685, 'f1': 0.5190377644446771}, 'rougeL': {'precision': 0.5752252865024408, 'recall': 0.7435683330393315, 'f1': 0.6138132056791251}}
BERT Score: {'precision': 0.9240190386772156, 'recall': 0.9471586346626282, 'f1': 0.9350123405456543}
gpt_in_context_results.xlsx
output/chat/exl/gpt_in_context_results.xlsx


...

ROUGE Score: {'rouge1': {'precision': 0.5811339246031405, 'recall': 0.7089040733463715, 'f1': 0.5557318329385577}, 'rouge2': {'precision': 0.4391083523696075, 'recall': 0.47188758447904466, 'f1': 0.3948137568922707}, 'rougeL': {'precision': 0.528140939261444, 'recall': 0.6191330576420973, 'f1': 0.4981295322715515}}
BERT Score: {'precision': 0.8897323608398438, 'recall': 0.9216877818107605, 'f1': 0.9047080278396606}
llama_in_context_results.xlsx
output/chat/exl/llama_in_context_results.xlsx


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROUGE Score: {'rouge1': {'precision': 0.5108540475400134, 'recall': 0.8282461078354025, 'f1': 0.5882730096084126}, 'rouge2': {'precision': 0.37974820429727274, 'recall': 0.6024653361333546, 'f1': 0.4335418212332945}, 'rougeL': {'precision': 0.4581774002556515, 'recall': 0.7405444892047938, 'f1': 0.5275838590424106}}
BERT Score: {'precision': 0.9005054831504822, 'recall': 0.9395279288291931, 'f1': 0.9191817045211792}
llama_results.xlsx
output/chat/exl/llama_results.xlsx


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROUGE Score: {'rouge1': {'precision': 0.4965818314325649, 'recall': 0.8441693723201981, 'f1': 0.5810611777108666}, 'rouge2': {'precision': 0.3722644493089244, 'recall': 0.6151298212189595, 'f1': 0.4317980799365111}, 'rougeL': {'precision': 0.44554259919126094, 'recall': 0.7550093042394351, 'f1': 0.5215252137579006}}
BERT Score: {'precision': 0.8979816436767578, 'recall': 0.940869927406311, 'f1': 0.9184987545013428}


In [11]:
read_chat_excel()

output/chat/exl/gimma_in_context_results.xlsx


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROUGE Score: {'rouge1': {'precision': 0.3319760486145554, 'recall': 0.41611023762661975, 'f1': 0.3455236298250166}, 'rouge2': {'precision': 0.171030514975807, 'recall': 0.20683018430580502, 'f1': 0.17777438644373544}, 'rougeL': {'precision': 0.27895362422568076, 'recall': 0.35063148669459643, 'f1': 0.29100514779838804}}
BERT Score: {'precision': 0.8736419677734375, 'recall': 0.8852952718734741, 'f1': 0.8791053295135498}
output/chat/exl/gimma_results.xlsx


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROUGE Score: {'rouge1': {'precision': 0.4020725646643636, 'recall': 0.5028998407226015, 'f1': 0.4181197273409562}, 'rouge2': {'precision': 0.25008226249023424, 'recall': 0.2947236320660153, 'f1': 0.25573554932495834}, 'rougeL': {'precision': 0.34978901306587057, 'recall': 0.4355266326036848, 'f1': 0.3633813440632038}}
BERT Score: {'precision': 0.8859197497367859, 'recall': 0.9004956483840942, 'f1': 0.8928059339523315}
output/chat/exl/gpt_in_context_results.xlsx


...

ROUGE Score: {'rouge1': {'precision': 0.0110453295986494, 'recall': 0.2180952380952381, 'f1': 0.019828171034090368}, 'rouge2': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}, 'rougeL': {'precision': 0.0110453295986494, 'recall': 0.2180952380952381, 'f1': 0.019828171034090368}}
BERT Score: {'precision': 0.7760744690895081, 'recall': 0.7491779923439026, 'f1': 0.7621397972106934}
output/chat/exl/gpt_results.xlsx


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROUGE Score: {'rouge1': {'precision': 0.5811339246031405, 'recall': 0.7089040733463715, 'f1': 0.5557318329385577}, 'rouge2': {'precision': 0.4391083523696075, 'recall': 0.47188758447904466, 'f1': 0.3948137568922707}, 'rougeL': {'precision': 0.528140939261444, 'recall': 0.6191330576420973, 'f1': 0.4981295322715515}}
BERT Score: {'precision': 0.8897323608398438, 'recall': 0.9216877818107605, 'f1': 0.9047080874443054}
output/chat/exl/llama_in_context_results.xlsx


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROUGE Score: {'rouge1': {'precision': 0.5108540475400134, 'recall': 0.8282461078354025, 'f1': 0.5882730096084126}, 'rouge2': {'precision': 0.37974820429727274, 'recall': 0.6024653361333546, 'f1': 0.4335418212332945}, 'rougeL': {'precision': 0.4581774002556515, 'recall': 0.7405444892047938, 'f1': 0.5275838590424106}}
BERT Score: {'precision': 0.9005054831504822, 'recall': 0.9395279288291931, 'f1': 0.9191817045211792}
'reference'


In [12]:
read_chat_excel()

gimma_in_context_results.xlsx
output/chat/exl/gimma_in_context_results.xlsx


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROUGE Score: {'rouge1': {'precision': 0.3319760486145554, 'recall': 0.41611023762661975, 'f1': 0.3455236298250166}, 'rouge2': {'precision': 0.171030514975807, 'recall': 0.20683018430580502, 'f1': 0.17777438644373544}, 'rougeL': {'precision': 0.27895362422568076, 'recall': 0.35063148669459643, 'f1': 0.29100514779838804}}
BERT Score: {'precision': 0.8736419677734375, 'recall': 0.8852952718734741, 'f1': 0.8791053891181946}
gimma_results.xlsx
output/chat/exl/gimma_results.xlsx


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROUGE Score: {'rouge1': {'precision': 0.4020725646643636, 'recall': 0.5028998407226015, 'f1': 0.4181197273409562}, 'rouge2': {'precision': 0.25008226249023424, 'recall': 0.2947236320660153, 'f1': 0.25573554932495834}, 'rougeL': {'precision': 0.34978901306587057, 'recall': 0.4355266326036848, 'f1': 0.3633813440632038}}
BERT Score: {'precision': 0.8859198689460754, 'recall': 0.9004956483840942, 'f1': 0.8928059339523315}
gpt_in_context_results.xlsx
output/chat/exl/gpt_in_context_results.xlsx


...

ROUGE Score: {'rouge1': {'precision': 0.5811339246031405, 'recall': 0.7089040733463715, 'f1': 0.5557318329385577}, 'rouge2': {'precision': 0.4391083523696075, 'recall': 0.47188758447904466, 'f1': 0.3948137568922707}, 'rougeL': {'precision': 0.528140939261444, 'recall': 0.6191330576420973, 'f1': 0.4981295322715515}}
BERT Score: {'precision': 0.8897323608398438, 'recall': 0.9216877818107605, 'f1': 0.9047080278396606}
llama_in_context_results.xlsx
output/chat/exl/llama_in_context_results.xlsx


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROUGE Score: {'rouge1': {'precision': 0.5108540475400134, 'recall': 0.8282461078354025, 'f1': 0.5882730096084126}, 'rouge2': {'precision': 0.37974820429727274, 'recall': 0.6024653361333546, 'f1': 0.4335418212332945}, 'rougeL': {'precision': 0.4581774002556515, 'recall': 0.7405444892047938, 'f1': 0.5275838590424106}}
BERT Score: {'precision': 0.9005054831504822, 'recall': 0.9395279288291931, 'f1': 0.9191817045211792}
llama_results.xlsx
output/chat/exl/llama_results.xlsx


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROUGE Score: {'rouge1': {'precision': 0.4965818314325649, 'recall': 0.8441693723201981, 'f1': 0.5810611777108666}, 'rouge2': {'precision': 0.3722644493089244, 'recall': 0.6151298212189595, 'f1': 0.4317980799365111}, 'rougeL': {'precision': 0.44554259919126094, 'recall': 0.7550093042394351, 'f1': 0.5215252137579006}}
BERT Score: {'precision': 0.8979816436767578, 'recall': 0.940869927406311, 'f1': 0.9184987545013428}


In [8]:
read_chat_excel()

gimma_in_context_results.xlsx
output/chat/exl/gimma_in_context_results.xlsx


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROUGE Score: {'rouge1': {'precision': 0.6369874710682439, 'recall': 0.7938519994414707, 'f1': 0.6745069094604318}, 'rouge2': {'precision': 0.4944676468510959, 'recall': 0.611390967990908, 'f1': 0.5218921465757874}, 'rougeL': {'precision': 0.5847008922385354, 'recall': 0.7278796494276069, 'f1': 0.619418469667238}}
BERT Score: {'precision': 0.927574634552002, 'recall': 0.9456157088279724, 'f1': 0.9361273050308228}
gimma_results.xlsx
output/chat/exl/gimma_results.xlsx


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROUGE Score: {'rouge1': {'precision': 0.6266314555759909, 'recall': 0.8107653896875779, 'f1': 0.6687772382424819}, 'rouge2': {'precision': 0.4909696037800951, 'recall': 0.6269287601399685, 'f1': 0.5190377644446771}, 'rougeL': {'precision': 0.5752252865024408, 'recall': 0.7435683330393315, 'f1': 0.6138132056791251}}
BERT Score: {'precision': 0.9240190386772156, 'recall': 0.9471586346626282, 'f1': 0.9350123405456543}
gpt_in_context_results.xlsx
output/chat/exl/gpt_in_context_results.xlsx


...

ROUGE Score: {'rouge1': {'precision': 0.5811339246031405, 'recall': 0.7089040733463715, 'f1': 0.5557318329385577}, 'rouge2': {'precision': 0.4391083523696075, 'recall': 0.47188758447904466, 'f1': 0.3948137568922707}, 'rougeL': {'precision': 0.528140939261444, 'recall': 0.6191330576420973, 'f1': 0.4981295322715515}}
BERT Score: {'precision': 0.8897323608398438, 'recall': 0.9216877818107605, 'f1': 0.9047080278396606}
llama_in_context_results.xlsx
output/chat/exl/llama_in_context_results.xlsx


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROUGE Score: {'rouge1': {'precision': 0.5108540475400134, 'recall': 0.8282461078354025, 'f1': 0.5882730096084126}, 'rouge2': {'precision': 0.37974820429727274, 'recall': 0.6024653361333546, 'f1': 0.4335418212332945}, 'rougeL': {'precision': 0.4581774002556515, 'recall': 0.7405444892047938, 'f1': 0.5275838590424106}}
BERT Score: {'precision': 0.9005054831504822, 'recall': 0.9395279288291931, 'f1': 0.9191817045211792}
llama_results.xlsx
output/chat/exl/llama_results.xlsx


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROUGE Score: {'rouge1': {'precision': 0.4965818314325649, 'recall': 0.8441693723201981, 'f1': 0.5810611777108666}, 'rouge2': {'precision': 0.3722644493089244, 'recall': 0.6151298212189595, 'f1': 0.4317980799365111}, 'rougeL': {'precision': 0.44554259919126094, 'recall': 0.7550093042394351, 'f1': 0.5215252137579006}}
BERT Score: {'precision': 0.8979816436767578, 'recall': 0.940869927406311, 'f1': 0.9184987545013428}


In [9]:
read_chat_excel()

gimma_in_context_results.xlsx
output/chat/exl/gimma_in_context_results.xlsx


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROUGE Score: {'rouge1': {'precision': 0.6369874710682439, 'recall': 0.7938519994414707, 'f1': 0.6745069094604318}, 'rouge2': {'precision': 0.4944676468510959, 'recall': 0.611390967990908, 'f1': 0.5218921465757874}, 'rougeL': {'precision': 0.5847008922385354, 'recall': 0.7278796494276069, 'f1': 0.619418469667238}}
BERT Score: {'precision': 0.927574634552002, 'recall': 0.9456157088279724, 'f1': 0.9361273050308228}
gimma_results.xlsx
output/chat/exl/gimma_results.xlsx


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROUGE Score: {'rouge1': {'precision': 0.6266314555759909, 'recall': 0.8107653896875779, 'f1': 0.6687772382424819}, 'rouge2': {'precision': 0.4909696037800951, 'recall': 0.6269287601399685, 'f1': 0.5190377644446771}, 'rougeL': {'precision': 0.5752252865024408, 'recall': 0.7435683330393315, 'f1': 0.6138132056791251}}
BERT Score: {'precision': 0.9240190386772156, 'recall': 0.9471586346626282, 'f1': 0.9350123405456543}
gpt_in_context_results.xlsx
output/chat/exl/gpt_in_context_results.xlsx


...

ROUGE Score: {'rouge1': {'precision': 0.5811339246031405, 'recall': 0.7089040733463715, 'f1': 0.5557318329385577}, 'rouge2': {'precision': 0.4391083523696075, 'recall': 0.47188758447904466, 'f1': 0.3948137568922707}, 'rougeL': {'precision': 0.528140939261444, 'recall': 0.6191330576420973, 'f1': 0.4981295322715515}}
BERT Score: {'precision': 0.8897323608398438, 'recall': 0.9216877818107605, 'f1': 0.9047080278396606}
llama_in_context_results.xlsx
output/chat/exl/llama_in_context_results.xlsx


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROUGE Score: {'rouge1': {'precision': 0.5108540475400134, 'recall': 0.8282461078354025, 'f1': 0.5882730096084126}, 'rouge2': {'precision': 0.37974820429727274, 'recall': 0.6024653361333546, 'f1': 0.4335418212332945}, 'rougeL': {'precision': 0.4581774002556515, 'recall': 0.7405444892047938, 'f1': 0.5275838590424106}}
BERT Score: {'precision': 0.9005054831504822, 'recall': 0.9395279288291931, 'f1': 0.9191817045211792}
llama_results.xlsx
output/chat/exl/llama_results.xlsx


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROUGE Score: {'rouge1': {'precision': 0.4965818314325649, 'recall': 0.8441693723201981, 'f1': 0.5810611777108666}, 'rouge2': {'precision': 0.3722644493089244, 'recall': 0.6151298212189595, 'f1': 0.4317980799365111}, 'rougeL': {'precision': 0.44554259919126094, 'recall': 0.7550093042394351, 'f1': 0.5215252137579006}}
BERT Score: {'precision': 0.8979816436767578, 'recall': 0.940869927406311, 'f1': 0.9184987545013428}
